# Fundamentals 00.4 - Runtime vLLM Provider API

Objetivo: probar la ruta `vllm-runtime` de forma aislada antes de usar agentes, systems o graphs con un modelo local/GPU.

Este notebook ensena la capa provider para vLLM. vLLM no es framework: es infraestructura externa que expone una API OpenAI-compatible. Agentic Systems se conecta a esa API con `provider="vllm-runtime"`.

Regla de diseno:

```text
Agentic Systems define el contrato de ejecucion.
vllm-runtime define el backend OpenAI-compatible.
vLLM server corre fuera de la libreria, normalmente en Colab/GPU.
```


## 0) Instalacion e imports minimos

En notebooks usa `%pip`, no `%%python -m pip`.

- `%pip` instala en el kernel activo donde despues haces `import agentic_systems`.
- `%%python` ejecuta otro proceso Python; puede instalar en un entorno distinto y luego el import falla.

Para Colab/PyPI:

```python
%pip install -U pip
%pip install -U "agentic-systems[all]"
```

`agentic-systems[all]` instala clientes e integraciones soportadas por la libreria. El servidor GPU `vllm` se instala aparte porque es infraestructura pesada y depende del runtime GPU.


In [ ]:
# Ejecuta esta celda solo en Colab o en un ambiente donde el paquete no este instalado.
# En notebooks, %pip instala en el kernel activo. No uses %%python para instalar dependencias.
#
# %pip install -U pip
# %pip install -U "agentic-systems[all]"


In [ ]:
import importlib.util
import json
import os
import subprocess
import time
from urllib.request import Request, urlopen

import agentic_systems as toolkit

print("agentic_systems:", toolkit.__name__)


## 1) Configuracion del provider

`vllm-runtime` lee configuracion desde variables de entorno o `.env`:

| Variable | Uso |
|---|---|
| `VLLM_BASE_URL` | URL OpenAI-compatible del servidor vLLM. |
| `VLLM_MODEL` | Modelo servido por vLLM. |
| `VLLM_API_KEY` | API key para servidores compatibles; normalmente `EMPTY` local. |

El default local es `http://127.0.0.1:8000/v1` y modelo `Qwen/Qwen3-0.6B`. El notebook no guarda secretos.


In [ ]:
# Valores seguros para Colab/local si todav?a no estan definidos.
os.environ.setdefault("VLLM_BASE_URL", "http://127.0.0.1:8000/v1")
os.environ.setdefault("VLLM_MODEL", "Qwen/Qwen3-0.6B")
os.environ.setdefault("VLLM_API_KEY", "EMPTY")

vllm_env = {
    "VLLM_BASE_URL": os.getenv("VLLM_BASE_URL"),
    "VLLM_MODEL": os.getenv("VLLM_MODEL"),
    "VLLM_API_KEY_configured": bool(os.getenv("VLLM_API_KEY")),
}

toolkit.show(vllm_env, title="Configuraci?n vLLM segura")


## 2) Declarar `RuntimeConfig`

`toolkit.runtime(provider="vllm-runtime")` no ejecuta el modelo. Solo declara el contrato de provider, modelo, scheduler y metadata segura.

`runtime.describe()` sirve para confirmar que leera Agentic Systems antes de hacer inferencia.


In [ ]:
scheduler = toolkit.scheduler(
    timeout_s=60,
    max_retries=0,
    max_tool_calls=4,
    max_turns=4,
    max_concurrency=1,
)

vllm_runtime = toolkit.runtime(
    provider="vllm-runtime",
    scheduler=scheduler,
)

auto_runtime = toolkit.runtime(
    provider="auto",
    scheduler=scheduler,
)

toolkit.show(vllm_runtime.describe(), title="vLLM runtime - describe")
toolkit.show(auto_runtime.describe(), title="Auto runtime - describe")


## 3) Servidor vLLM opcional en Colab/GPU

`agentic-systems` no instala ni arranca el servidor GPU `vllm`. La libreria solo habla con una API OpenAI-compatible ya levantada.

Si estas en Colab/GPU y quieres levantar Qwen localmente, instala `vllm` en el ambiente y arranca el servidor en background. No uses `%%python -m vllm...` para esto: esa forma bloquea la celda y falla si `vllm` no esta instalado.

Comandos equivalentes para Colab:

```bash
pip install -U "vllm>=0.9.0" "openai>=1.0.0"
python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen3-0.6B \
  --served-model-name Qwen/Qwen3-0.6B \
  --host 127.0.0.1 \
  --port 8000
```

La siguiente celda es segura: si `vllm` no esta instalado, reporta `skipped`; si ya hay servidor levantado, no lo duplica.


In [ ]:
def wait_for_vllm_server(base_url: str, *, timeout_s: int = 60) -> dict:
    models_url = base_url.rstrip("/") + "/models"
    deadline = time.time() + timeout_s
    last_error = None
    while time.time() < deadline:
        try:
            request = Request(models_url, headers={"Authorization": f"Bearer {os.getenv('VLLM_API_KEY', 'EMPTY')}"})
            with urlopen(request, timeout=3) as response:
                payload = json.loads(response.read().decode("utf-8"))
            return {"status": "ok", "models_url": models_url, "response": payload}
        except Exception as exc:
            last_error = str(exc)
            time.sleep(2)
    return {"status": "skipped", "models_url": models_url, "reason": last_error or "timeout"}


def start_vllm_server_if_available(*, model: str, host: str = "127.0.0.1", port: int = 8000) -> dict:
    if importlib.util.find_spec("vllm") is None:
        return {
            "status": "skipped",
            "reason": "vllm no esta instalado. En Colab usa: pip install -U 'vllm>=0.9.0'",
        }

    health = wait_for_vllm_server(os.getenv("VLLM_BASE_URL", f"http://{host}:{port}/v1"), timeout_s=3)
    if health["status"] == "ok":
        return {"status": "already_running", "health": health}

    cmd = [
        "python",
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        model,
        "--served-model-name",
        model,
        "--host",
        host,
        "--port",
        str(port),
    ]
    process = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    health = wait_for_vllm_server(f"http://{host}:{port}/v1", timeout_s=90)
    return {"status": "started" if health["status"] == "ok" else "starting_or_failed", "pid": process.pid, "health": health}


# Cambia a True solo si quieres que este notebook levante el servidor local/GPU.
START_LOCAL_VLLM_SERVER = False

if START_LOCAL_VLLM_SERVER:
    server_status = start_vllm_server_if_available(model=os.getenv("VLLM_MODEL", "Qwen/Qwen3-0.6B"))
else:
    server_status = {"status": "skipped", "reason": "START_LOCAL_VLLM_SERVER=False"}

toolkit.show(server_status, title="vLLM server launch opcional")


## 4) Health check opcional del servidor

Esta celda intenta consultar `/models` en el servidor vLLM OpenAI-compatible. Si no hay servidor levantado, no falla el notebook: reporta `skipped`.


In [ ]:
def vllm_models_url(base_url: str) -> str:
    return base_url.rstrip("/") + "/models"

models_url = vllm_models_url(os.getenv("VLLM_BASE_URL", "http://127.0.0.1:8000/v1"))

try:
    request = Request(models_url, headers={"Authorization": f"Bearer {os.getenv('VLLM_API_KEY', 'EMPTY')}"})
    with urlopen(request, timeout=3) as response:
        payload = json.loads(response.read().decode("utf-8"))
    toolkit.show({"status": "ok", "models_url": models_url, "response": payload}, title="vLLM server health")
    VLLM_SERVER_AVAILABLE = True
except Exception as exc:
    toolkit.show({"status": "skipped", "models_url": models_url, "reason": str(exc)}, title="vLLM server health")
    VLLM_SERVER_AVAILABLE = False


## 5) Tool smoke opcional con `vllm-runtime`

Este smoke usa una tool normal de Agentic Systems. Si el servidor vLLM esta disponible y soporta tool calling compatible, el agente puede llamarla.

Si el servidor no esta levantado, la celda reporta `skipped` para que el notebook siga siendo portable en local, VSCode y Colab.


In [ ]:
@toolkit.tool
def sumar(a: int, b: int) -> dict:
    """Suma dos enteros."""
    return {"result": a + b}

policy = toolkit.RunPolicy(
    max_turns=4,
    max_tool_calls=2,
    temperature=0.0,
    tool_choice="auto",
    repair=True,
    max_repairs=1,
    trace="compact",
    strict=True,
)

if not VLLM_SERVER_AVAILABLE:
    toolkit.show({"status": "skipped", "reason": "Servidor vLLM no disponible."}, title="Tool smoke vLLM")
    result = None
else:
    system = toolkit.AgenticSystem(runtime=vllm_runtime)
    agent = system.agent(
        name="qwen_calculator",
        instructions="Usa la tool sumar para resolver la petici?n y responde breve.",
        tools=[sumar],
        engine="vllm-runtime",
        runtime=vllm_runtime,
        policy=policy,
    )
    result = agent.run("Suma 10 y 20 usando la tool sumar.")
    toolkit.human_result(result)


## 6) Cierre de API

Este notebook cubre la ruta provider vLLM sin introducir frameworks externos. La composicion con `AgenticSystem`, LangGraph, Strands u OpenAI Agents se ensena en notebooks posteriores.


In [ ]:
api_coverage = [
    {"api": "toolkit.runtime(provider='vllm-runtime')", "description": "Declara vLLM como provider canonico."},
    {"api": "toolkit.runtime(provider='auto')", "description": "Selecciona vLLM automaticamente cuando VLLM_BASE_URL esta configurado."},
    {"api": "RuntimeConfig.describe", "description": "Muestra resolucion y configuracion segura."},
    {"api": "toolkit.scheduler", "description": "Declara limites de ejecucion."},
    {"api": "toolkit.RunPolicy", "description": "Controla loops, tool calls y trazas."},
    {"api": "toolkit.human_result", "description": "Renderiza resultados reales de ejecucion."},
    "VLLM_RUNTIME_ENGINE",
]

toolkit.show({"notebook": "00_runtime_vllm_provider_api.ipynb", "api_coverage": api_coverage}, title="API coverage")
